# Section 2 — Numerical Error and Floating-Point Arithmetic

[← Taylor Series](01_taylor_series.ipynb) ·
[Module 1](README.md) ·
[Next: Bisection →](03_bisection_method.ipynb)

Computers do not store arbitrary real numbers exactly. Numerical analysis
therefore tracks both mathematical approximation error and error introduced by
finite-precision arithmetic.

### Learning goals

- compute absolute and relative error;
- distinguish rounding from chopping;
- inspect IEEE-754 binary64 fields;
- find the next representable floating-point number;
- understand unit-in-the-last-place (ULP) spacing;
- recognize loss of significance;
- reorganize a formula to reduce numerical error;
- solve inequalities involving a prescribed relative-error tolerance.


In [ ]:
import math
import struct


## Absolute and relative error

If $p$ is exact and $p^\ast$ is an approximation,

$$
\text{absolute error}=|p-p^\ast|,
\qquad
\text{relative error}=\frac{|p-p^\ast|}{|p|}.
$$


In [ ]:
exact_value = math.pi
approximation = 3.14

absolute_error = abs(exact_value - approximation)
relative_error = absolute_error / abs(exact_value)

print(f"absolute error = {absolute_error:.10f}")
print(f"relative error = {relative_error:.10e}")


## Three-digit chopping and rounding

Small-digit arithmetic makes finite-precision effects visible.


In [ ]:
def chop(value, digits):
    """Chop a decimal value to a fixed number of significant digits."""
    if value == 0:
        return 0.0

    exponent = math.floor(math.log10(abs(value)))
    scale = 10 ** (digits - 1 - exponent)
    return math.trunc(value * scale) / scale


def round_sig(value, digits):
    """Round a decimal value to a fixed number of significant digits."""
    if value == 0:
        return 0.0

    exponent = math.floor(math.log10(abs(value)))
    decimals = digits - 1 - exponent
    return round(value, decimals)


a = 58.5
b = -0.123
c = 1.96

left_to_right = chop(chop(a + b, 3) + c, 3)
reorganized = chop(a + chop(b + c, 3), 3)
exact_value = a + b + c

print(f"exact value         = {exact_value}")
print(f"left-to-right chop  = {left_to_right}")
print(f"reorganized chop    = {reorganized}")


## IEEE-754 binary64

Python's ordinary `float` is normally IEEE-754 binary64:

- 1 sign bit;
- 11 exponent bits;
- 52 fraction bits.

For a normalized number,

$$
(-1)^s 2^{c-1023}(1+f).
$$


In [ ]:
def binary64_fields(value):
    """Return sign, exponent, and fraction fields for a Python float."""
    packed = struct.pack(">d", value)
    integer = struct.unpack(">Q", packed)[0]
    bits = f"{integer:064b}"
    return bits[0], bits[1:12], bits[12:]


value = -27.0
sign, exponent, fraction = binary64_fields(value)

print(f"value    = {value}")
print(f"sign     = {sign}")
print(f"exponent = {exponent}")
print(f"fraction = {fraction}")


## The next machine number and ULP spacing

Floating-point numbers are discrete. If $x$ is a machine number, the **next
representable number** above it is not $x+\varepsilon$ for an arbitrary tiny
real $\varepsilon$; it is the next binary64 pattern.

Python exposes this directly with `math.nextafter`.

The gap

$$
\operatorname{ulp}(x)
$$

is the spacing between nearby representable numbers at the scale of $x$.
This is the computational version of the Exam 1 question asking for the
smallest machine number greater than a given binary64 number.


In [ ]:
x_value = 1.0
next_value = math.nextafter(x_value, math.inf)
spacing = math.ulp(x_value)

print(f"x                = {x_value:.17g}")
print(f"next float       = {next_value:.17g}")
print(f"difference       = {next_value - x_value:.17g}")
print(f"math.ulp(x)      = {spacing:.17g}")


## Catastrophic cancellation

Subtracting nearly equal floating-point numbers can destroy significant digits.

For small $c>0$,

$$
\sqrt{x^2+c}-x
$$

subtracts nearby numbers. Rationalizing gives

$$
\frac{c}{\sqrt{x^2+c}+x}.
$$


In [ ]:
x_value = 1.0e8
c_value = 1.0

unstable = math.sqrt(x_value**2 + c_value) - x_value
stable = c_value / (math.sqrt(x_value**2 + c_value) + x_value)

print(f"direct formula       = {unstable:.17g}")
print(f"rationalized formula = {stable:.17g}")


## Relative-error intervals

A review problem asks for all approximations $p^\ast$ to $p=850$ having relative
error at most $10^{-3}$.

Start from the definition:

$$
\frac{|p-p^\ast|}{|p|}\le10^{-3}.
$$

Thus

$$
|850-p^\ast|\le850(10^{-3})=0.85,
$$

so

$$
\boxed{849.15\le p^\ast\le850.85}.
$$

The important step is to turn a relative-error requirement into an absolute
interval around the true value.


In [ ]:
p = 850.0
relative_tolerance = 1.0e-3

radius = abs(p) * relative_tolerance
lower = p - radius
upper = p + radius

print(f"allowable interval = [{lower:.2f}, {upper:.2f}]")


### References

- Leon Q. Brin, *Tea Time Numerical Analysis*.
- Richard L. Burden and J. Douglas Faires, *Numerical Analysis*.
- MAD 4401 Exam 1 review and course notes.

[← Taylor Series](01_taylor_series.ipynb) ·
[Next: Bisection →](03_bisection_method.ipynb)
